#Download dataset

In [0]:
%%time
import os
os.environ['KAGGLE_USERNAME'] = "smurli" 
os.environ['KAGGLE_KEY'] = "660e4133e04de51c0138a516db115cc6" # key from the json file
!kaggle competitions download -c plant-seedlings-classification

  0% 0.00/5.13k [00:00<?, ?B/s]
100% 5.13k/5.13k [00:00<00:00, 7.00MB/s]
 85% 73.0M/86.0M [00:01<00:00, 32.0MB/s]
100% 86.0M/86.0M [00:01<00:00, 52.5MB/s]
 99% 1.58G/1.60G [00:28<00:00, 71.5MB/s]
100% 1.60G/1.60G [00:29<00:00, 59.1MB/s]
CPU times: user 292 ms, sys: 59.8 ms, total: 352 ms
Wall time: 36.9 s


In [0]:
!unzip -q test.zip
!unzip -q train.zip

# Create Train and Test Datset

In [0]:
import glob
import cv2
import numpy as np
from tqdm import tqdm

In [0]:
TRAIN_PATH="/content/train"
TEST_PATH="/content/test"
SIZE=64

In [0]:
fileList = glob.glob(TRAIN_PATH+"/*/*.png")

images = []
labels = []
for file in tqdm(fileList):
    images.append(cv2.resize(cv2.imread(file),(SIZE,SIZE)))
    labels.append(file.split('/')[-2])

trainImages = np.asarray(images)
trainLabels = np.asarray(labels)

100%|██████████| 4750/4750 [00:31<00:00, 150.68it/s]


In [0]:
#normalize imaes
trainImages = trainImages/255

In [0]:
from sklearn.preprocessing import LabelEncoder
import tensorflow.keras as keras 

labelEncoder = LabelEncoder()
labelsEnc = labelEncoder.fit_transform(labels)

labelsCat = keras.utils.to_categorical(labelsEnc)

In [0]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(trainImages,labelsCat,test_size=0.2,random_state=5)

print(X_train.shape,X_test.shape,y_train.shape,y_test.shape)

(3800, 64, 64, 3) (950, 64, 64, 3) (3800, 12) (950, 12)


In [0]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import BatchNormalization

def get_model():
    model = Sequential()

    model.add(Conv2D(filters=64, kernel_size=(5, 5), input_shape=(SIZE, SIZE, 3), activation='relu'))
    model.add(BatchNormalization(axis=3))
    model.add(Conv2D(filters=64, kernel_size=(5, 5), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization(axis=3))
    model.add(Dropout(0.1))

    model.add(Conv2D(filters=128, kernel_size=(5, 5), activation='relu'))
    model.add(BatchNormalization(axis=3))
    model.add(Conv2D(filters=128, kernel_size=(5, 5), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization(axis=3))
    model.add(Dropout(0.1))

    model.add(Conv2D(filters=256, kernel_size=(5, 5), activation='relu'))
    model.add(BatchNormalization(axis=3))
    model.add(Conv2D(filters=256, kernel_size=(5, 5), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization(axis=3))
    model.add(Dropout(0.1))

    model.add(Flatten())

    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))

    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))

    model.add(Dense(12, activation='softmax'))
    
    return model

In [0]:
model = get_model()
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [0]:
model.fit(X_train, y_train, batch_size=64, validation_data=(X_test,y_test), epochs=50)

Train on 3800 samples, validate on 950 samples
Epoch 1/50
3800/3800 [==============================] - 10s 3ms/sample - loss: 2.5287 - acc: 0.3021 - val_loss: 8.8044 - val_acc: 0.1379
Epoch 2/50
3800/3800 [==============================] - 7s 2ms/sample - loss: 1.6297 - acc: 0.5008 - val_loss: 11.2894 - val_acc: 0.1379
Epoch 3/50
3800/3800 [==============================] - 7s 2ms/sample - loss: 1.2426 - acc: 0.6103 - val_loss: 12.1662 - val_acc: 0.1379
Epoch 4/50
3800/3800 [==============================] - 7s 2ms/sample - loss: 1.0615 - acc: 0.6655 - val_loss: 11.8401 - val_acc: 0.1474
Epoch 5/50
3800/3800 [==============================] - 7s 2ms/sample - loss: 0.8183 - acc: 0.7250 - val_loss: 8.0875 - val_acc: 0.1579
Epoch 6/50
3800/3800 [==============================] - 7s 2ms/sample - loss: 0.7153 - acc: 0.7582 - val_loss: 7.9332 - val_acc: 0.1916
Epoch 7/50
3800/3800 [==============================] - 7s 2ms/sample - loss: 0.6543 - acc: 0.7863 - val_loss: 6.7763 - val_acc: 0.16

In [0]:
y_pred = model.predict(X_test)
y_class = np.argmax(y_pred, axis = 1) 
y_check = np.argmax(y_test, axis = 1) 

print("Accuracy of Test Dataset:")
cmatrix = confusion_matrix(y_check, y_class)
print(cmatrix)
print(classification_report(y_check, y_class))

Accuracy of Test Dataset:
[[  9   0   0   0   0   2  45   0   1   0   0   0]
 [  0  63   0   0   0   3   0   1   5   4   1   3]
 [  0   0  12   0   0   6   0   1  26   2   4   3]
 [  1   0   0 112   0   1   1   0   2   0   0   0]
 [  1   0   0   1  20   2  12   0   7   1   0   0]
 [  0   0   0   1   0  89   3   1   6   1   1   1]
 [  2   0   0   0   0   1 125   1   1   0   1   0]
 [  0   0   0   3   0   0   0  46   2   1   0   0]
 [  0   0   0   3   0   0   1   1  91   0   0   0]
 [  0   0   0   7   0   0   0   0   6  27   0   0]
 [  0   0   0   3   0   1   0   1   2   0  82   0]
 [  0   0   0   5   0   0   0   2   5   1   0  74]]
              precision    recall  f1-score   support

           0       0.69      0.16      0.26        57
           1       1.00      0.79      0.88        80
           2       1.00      0.22      0.36        54
           3       0.83      0.96      0.89       117
           4       1.00      0.45      0.62        44
           5       0.85      0.86   

In [0]:
y_pred = model.predict(X_train)
y_class = np.argmax(y_pred, axis = 1) 
y_check = np.argmax(y_train, axis = 1) 

cmatrix = confusion_matrix(y_check, y_class)
print("Accuracy of Train Dataset:")
print(cmatrix)
print(classification_report(y_check, y_class))

Accuracy of Train Dataset:
[[127   0   0   0   0   0  76   0   3   0   0   0]
 [  0 280   0   0   0   1   0   7  12   6   0   4]
 [  0   9  62   0   0  20   0   0 104   6   9  23]
 [  0   0   0 489   0   0   1   0   3   1   0   0]
 [ 14   0   0   0 120   2  27   0  10   1   1   2]
 [  0   0   0   0   0 347  11   2   7   0   1   4]
 [  0   0   0   0   0   1 521   0   1   0   0   0]
 [  0   0   0   0   0   0   2 166   0   1   0   0]
 [  0   0   0   0   0   0   1   0 419   0   0   0]
 [  0   0   0  10   0   0   0   0   8 173   0   0]
 [  0   0   0   4   0   0   0   4   1   0 398   0]
 [  0   0   0   7   0   0   0   1  10   1   0 279]]
              precision    recall  f1-score   support

           0       0.90      0.62      0.73       206
           1       0.97      0.90      0.93       310
           2       1.00      0.27      0.42       233
           3       0.96      0.99      0.97       494
           4       1.00      0.68      0.81       177
           5       0.94      0.93  

The model seems to be doing good on train data but the test data results are not so good and results seems be varying a lot between epochs. Looks like the model is overfitting or memorizing the data set.

# Adding Data Augmentation
Lets try out some data agumentation to make the model generalize better.

In [0]:
from keras.preprocessing.image import ImageDataGenerator

generator = ImageDataGenerator(rotation_range = 180,zoom_range = 0.1,width_shift_range = 0.1,height_shift_range = 0.1,horizontal_flip = True,vertical_flip = True)
generator.fit(X_train)

In [0]:
model2 = get_model()
model2.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [0]:
hist = model2.fit_generator(generator.flow(X_train, y_train, batch_size=64), 
                            epochs=50, validation_data=(X_test, y_test))


Epoch 1/50
60/60 [==============================] - 16s 271ms/step - loss: 2.6138 - acc: 0.2805 - val_loss: 12.4552 - val_acc: 0.1379
Epoch 2/50
60/60 [==============================] - 7s 125ms/step - loss: 1.9325 - acc: 0.4221 - val_loss: 20.8510 - val_acc: 0.1379
Epoch 3/50
60/60 [==============================] - 8s 129ms/step - loss: 1.5222 - acc: 0.5176 - val_loss: 15.1833 - val_acc: 0.1379
Epoch 4/50
60/60 [==============================] - 8s 128ms/step - loss: 1.3385 - acc: 0.5658 - val_loss: 15.2010 - val_acc: 0.1400
Epoch 5/50
60/60 [==============================] - 8s 130ms/step - loss: 1.1027 - acc: 0.6345 - val_loss: 12.1650 - val_acc: 0.1505
Epoch 6/50
60/60 [==============================] - 8s 129ms/step - loss: 0.9793 - acc: 0.6792 - val_loss: 13.1619 - val_acc: 0.1705
Epoch 7/50
60/60 [==============================] - 8s 129ms/step - loss: 0.8428 - acc: 0.7245 - val_loss: 7.4947 - val_acc: 0.1800
Epoch 8/50
60/60 [==============================] - 8s 129ms/step - l

In [0]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

y_pred = model2.predict(X_test)
y_class = np.argmax(y_pred, axis = 1) 
y_check = np.argmax(y_test, axis = 1) 

print("Accuracy of Test Dataset:")
cmatrix = confusion_matrix(y_check, y_class)
print(cmatrix)
print(classification_report(y_check, y_class))

Accuracy of Test Dataset:
[[ 37   0   0   0   7   2   6   0   1   0   0   4]
 [  0  76   0   0   0   0   0   3   0   0   1   0]
 [  0  42   9   0   0   0   0   1   1   0   0   1]
 [  0   0   0 104   0   0   0   4   1   0   8   0]
 [  0   4   0   0  28   1   1   1   2   0   0   7]
 [  0   1   1   0   0  93   0   3   2   0   0   3]
 [ 58   0   0   0   1   0  70   0   2   0   0   0]
 [  0   0   0   0   0   0   0  50   1   0   0   1]
 [  1   0   0   0   0   0   0   3  79   0   0  13]
 [  0   4   0   6   0   0   0   3   9  18   0   0]
 [  0   3   0   1   0   1   0   5   0   0  79   0]
 [  0   0   0   0   0   0   1  21   1   0   0  64]]
              precision    recall  f1-score   support

           0       0.39      0.65      0.48        57
           1       0.58      0.95      0.72        80
           2       0.90      0.17      0.28        54
           3       0.94      0.89      0.91       117
           4       0.78      0.64      0.70        44
           5       0.96      0.90   

In [0]:
y_pred = model2.predict(X_train)
y_class = np.argmax(y_pred, axis = 1) 
y_check = np.argmax(y_train, axis = 1) 

cmatrix = confusion_matrix(y_check, y_class)
print("Accuracy of Train Dataset:")
print(cmatrix)
print(classification_report(y_check, y_class))

Accuracy of Train Dataset:
[[150   0   0   0  10   2  34   2   1   0   0   7]
 [  0 300   0   0   0   0   0   9   0   0   0   1]
 [  0 186  35   2   1   0   0   1   3   0   0   5]
 [  0   1   0 443   0   0   0  14  14   2  19   1]
 [  4  10   3   0 125   4   2  11   8   0   0  10]
 [  0   1   0   0   0 344   2  12   0   0   2  11]
 [227   0   0   0   2   3 272   6   5   0   0   8]
 [  0   0   0   0   0   0   0 168   0   0   0   1]
 [  1   4   0   2   0   0   0   2 331   0   0  80]
 [  0  17   0  33   0   2   0  15  39  80   0   5]
 [  0   8   0   0   0   1   1  26   0   0 371   0]
 [  0   2   0   1   0   0   0  73   1   0   0 221]]
              precision    recall  f1-score   support

           0       0.39      0.73      0.51       206
           1       0.57      0.97      0.72       310
           2       0.92      0.15      0.26       233
           3       0.92      0.90      0.91       494
           4       0.91      0.71      0.79       177
           5       0.97      0.92  

Variance error seems to have reduced and the overall model accuracy has gone down as well.

# Improvise Model
Lets increase the model configuration and check if it giving better results.

In [0]:
def get_model2():
    model = Sequential()

    model.add(Conv2D(filters=64, kernel_size=(5, 5), input_shape=(SIZE, SIZE, 3), activation='relu'))
    model.add(BatchNormalization(axis=3))
    model.add(Conv2D(filters=64, kernel_size=(5, 5), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization(axis=3))
    model.add(Dropout(0.1))

    model.add(Conv2D(filters=128, kernel_size=(5, 5), activation='relu'))
    model.add(BatchNormalization(axis=3))
    model.add(Conv2D(filters=128, kernel_size=(5, 5), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization(axis=3))
    model.add(Dropout(0.1))

    model.add(Conv2D(filters=256, kernel_size=(5, 5), activation='relu'))
    model.add(BatchNormalization(axis=3))
    model.add(Conv2D(filters=256, kernel_size=(5, 5), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization(axis=3))
    model.add(Dropout(0.1))

    model.add(Flatten())

    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))

    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))

    model.add(Dense(12, activation='softmax'))
    
    return model

In [0]:
from keras.preprocessing.image import ImageDataGenerator

generator = ImageDataGenerator(rotation_range = 180,zoom_range = 0.1,width_shift_range = 0.1,height_shift_range = 0.1,horizontal_flip = True,vertical_flip = True)
generator.fit(X_train)

In [0]:
model3 = get_model2()
model3.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [0]:
hist = model3.fit_generator(generator.flow(X_train, y_train, batch_size=64), 
                            epochs=50, validation_data=(X_test, y_test))


Epoch 1/50
60/60 [==============================] - 16s 270ms/step - loss: 2.6214 - acc: 0.2679 - val_loss: 6.0388 - val_acc: 0.1379
Epoch 2/50
60/60 [==============================] - 8s 126ms/step - loss: 1.9480 - acc: 0.3945 - val_loss: 16.7921 - val_acc: 0.1379
Epoch 3/50
60/60 [==============================] - 8s 130ms/step - loss: 1.5642 - acc: 0.4987 - val_loss: 21.1137 - val_acc: 0.1379
Epoch 4/50
60/60 [==============================] - 8s 130ms/step - loss: 1.2699 - acc: 0.5905 - val_loss: 11.4029 - val_acc: 0.1611
Epoch 5/50
60/60 [==============================] - 8s 134ms/step - loss: 1.1382 - acc: 0.6276 - val_loss: 8.8449 - val_acc: 0.1632
Epoch 6/50
60/60 [==============================] - 8s 130ms/step - loss: 0.9760 - acc: 0.6795 - val_loss: 12.3034 - val_acc: 0.1579
Epoch 7/50
60/60 [==============================] - 8s 129ms/step - loss: 0.8901 - acc: 0.7018 - val_loss: 7.4157 - val_acc: 0.1768
Epoch 8/50
60/60 [==============================] - 8s 131ms/step - los

In [0]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

y_pred = model3.predict(X_test)
y_class = np.argmax(y_pred, axis = 1) 
y_check = np.argmax(y_test, axis = 1) 

cmatrix = confusion_matrix(y_check, y_class)
print("Accuracy of Test Dataset:")
print(cmatrix)
print(classification_report(y_check, y_class))

Accuracy of Test Dataset:
[[ 28   0   0   0   2   0  26   0   1   0   0   0]
 [  0  72   2   0   0   0   0   0   0   1   3   2]
 [  0   0  51   0   3   0   0   0   0   0   0   0]
 [  0   0   0 101   0   0   0   0   7   1   8   0]
 [  1   0   0   0  38   1   2   0   1   0   0   1]
 [  1   0   1   0   0  98   0   1   0   0   2   0]
 [ 12   0   0   0   0   0 118   0   1   0   0   0]
 [  0   0   0   0   1   1   0  48   2   0   0   0]
 [  0   0   0   0   0   0   0   0  93   2   0   1]
 [  0   0   1   4   0   1   0   0   2  27   5   0]
 [  0   0   0   0   0   0   0   1   0   0  88   0]
 [  0   0   0   1   1   0   0   1   1   0   0  83]]
              precision    recall  f1-score   support

           0       0.67      0.49      0.57        57
           1       1.00      0.90      0.95        80
           2       0.93      0.94      0.94        54
           3       0.95      0.86      0.91       117
           4       0.84      0.86      0.85        44
           5       0.97      0.95   

In [0]:
y_pred = model3.predict(X_train)
y_class = np.argmax(y_pred, axis = 1) 
y_check = np.argmax(y_train, axis = 1) 

cmatrix = confusion_matrix(y_check, y_class)
print("Accuracy of Train Dataset:")
print(cmatrix)
print(classification_report(y_check, y_class))

Accuracy of Train Dataset:
[[122   0   0   0   2   1  80   0   1   0   0   0]
 [  0 283  13   0   0   1   0   0   0   1  10   2]
 [  0   2 223   1   3   1   0   0   3   0   0   0]
 [  0   0   0 440   0   0   0   0  30   8  16   0]
 [  5   0   1   0 162   2   4   0   0   0   0   3]
 [  0   0   0   1   0 367   1   0   0   0   3   0]
 [ 48   0   0   0   2   1 468   0   4   0   0   0]
 [  0   0   0   0   0   1   2 166   0   0   0   0]
 [  3   0   0   2   0   0   1   0 411   0   1   2]
 [  0   0   1  19   0   0   0   0   4 154  13   0]
 [  0   0   1   0   0   1   2   0   0   0 403   0]
 [  0   0   0   1   0   2   0   1   2   0   0 292]]
              precision    recall  f1-score   support

           0       0.69      0.59      0.64       206
           1       0.99      0.91      0.95       310
           2       0.93      0.96      0.94       233
           3       0.95      0.89      0.92       494
           4       0.96      0.92      0.94       177
           5       0.97      0.99  

Improvised model seems to be giving better results in terms of bias and variance errors.